# DynamoDB 데이터 준비 - prepare_ddb

로컬 `conf/`와 `data/`를 DynamoDB 테이블 `gsretail-mlops-edu-hjsong`에 등록합니다.

## 1. 환경 설정

In [1]:
import sys
import yaml
import base64
import boto3
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd().parent / 'modeling'))
from ddb_store import DDBStore

BASE_DIR = Path.cwd()
CONF_DIR = BASE_DIR / 'conf'
DATA_DIR = BASE_DIR / 'data'
print(f'Base: {BASE_DIR}')

Base: /home/ec2-user/SageMaker/gs-ds-env/samples/hjsong/prepare_input


## 2. 설정 파일 로드

In [2]:
def load_yaml(p):
    with open(p, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

env_config   = load_yaml(CONF_DIR / 'env.yml')
meta_config  = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')
print('설정 로드:', meta_config['experiment'])

설정 로드: tuned-hjsong-v1


## 3. DynamoDB 키 구성

In [3]:
USER_ID    = meta_config['user_id']
PROJECT    = meta_config['project']
EXPERIMENT = meta_config['experiment']
VERSION    = meta_config['version']
ENV        = env_config['env']
REGION     = env_config['region']

store  = DDBStore(region=REGION)
EXP_PK = DDBStore.make_exp_pk(USER_ID, PROJECT, EXPERIMENT)
print(f'EXP_PK: {EXP_PK}')

EXP_PK: EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v1


## 4. 테이블 확인 / 생성

In [4]:
store.ensure_table_exists()

   ✅ 테이블 'gsretail-mlops-edu-hjsong' 존재 확인


True

In [6]:
import boto3                                                                        
   
client = boto3.client('dynamodb', region_name='us-east-1')                          
                                                                                  
resp = client.describe_table(TableName='gsretail-mlops-edu-hjsong')                 
t = resp['Table']                                                                   
                                                                                  
print('=== Key Schema ===')                                                         
for k in t['KeySchema']:                                                            
  print(f"  {k['AttributeName']} ({k['KeyType']})")                               
                                                                                  
print('\n=== Attribute Definitions ===')                                            
for a in t['AttributeDefinitions']:
  print(f"  {a['AttributeName']} : {a['AttributeType']}")                         
                                                                                  
print('\n=== GSI ===')
for g in t.get('GlobalSecondaryIndexes', []):                                       
  print(f"  [{g['IndexName']}]")                                                
  for k in g['KeySchema']:                                                        
      print(f"    {k['AttributeName']} ({k['KeyType']})")

=== Key Schema ===
  experiment_id (HASH)
  entity_type (RANGE)

=== Attribute Definitions ===
  entity_type : S
  experiment_id : S

=== GSI ===


## 5. META 등록

In [5]:
store.put_experiment_meta(
    exp_pk=EXP_PK, user_id=USER_ID, project=PROJECT,
    experiment=EXPERIMENT, version=VERSION, env=ENV, region=REGION,
)
print(f'[OK] META: {EXP_PK}')

ClientError: An error occurred (ValidationException) when calling the PutItem operation: One or more parameter values were invalid: Missing the key experiment_id in the item

## 6. CONF 등록 (env/meta/model YAML)

In [ ]:
store.put_experiment_conf(
    exp_pk=EXP_PK,
    env_config=env_config,
    meta_config=meta_config,
    model_config=model_config,
)
print(f'[OK] CONF: {EXP_PK}')

## 7. DATA#split 등록

In [ ]:
for split in ['train', 'validation', 'test']:
    csv_path  = DATA_DIR / f'{split}.csv'
    csv_bytes = csv_path.read_bytes()
    row_count = sum(1 for _ in open(csv_path)) - 1
    store.put_dataset_split(
        exp_pk=EXP_PK, split=split, version=VERSION,
        csv_bytes=csv_bytes, row_count=row_count,
    )
    print(f'  [OK] DATA#{split}: {row_count}rows, {len(csv_bytes)/1024:.1f}KB -> base64 {len(base64.b64encode(csv_bytes))/1024:.1f}KB')
print('데이터셋 등록 완료')

## 8. 검증

In [ ]:
ddb   = boto3.resource('dynamodb', region_name=REGION)
table = ddb.Table('gsretail-mlops-edu-hjsong')
all_ok = True
for sk in ['META','CONF','DATA#train','DATA#validation','DATA#test']:
    exists = 'Item' in table.get_item(Key={'PK': EXP_PK, 'SK': sk})
    if not exists: all_ok = False
    print(f'  {"OK" if exists else "MISSING"}: {sk}')
print('완료' if all_ok else '일부 누락')

## 9. 요약

In [ ]:
print(f'PK   : {EXP_PK}')
print('SKs  : META, CONF, DATA#train, DATA#validation, DATA#test')
print('Table: gsretail-mlops-edu-hjsong')
print('다음  : modeling_ddb.ipynb')